# load_openaire_researchproduct_pids

Prototipo del nodo `load_openaire_researchproduct_pids` del pipeline `load_openaire`. No guarda datasets.


In [ ]:
from datetime import date
import pandas as pd

%load_ext kedro.ipython


In [ ]:
df_researchproduct_raw = catalog.load('raw/openaire/researchproduct/parquet/researchproduct_dev')
df_researchproduct_raw.head(2)


In [ ]:
def _add_openaire_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    return df


In [ ]:
def _add_openaire_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = df.copy()
    if load_datetime is None:
        load_datetime = date.today()
    df["_load_datetime"] = load_datetime
    return df


In [ ]:
def load_openaire_researchproduct_pids(df: pd.DataFrame)-> pd.DataFrame:
    df = _add_openaire_extracted_metadata(df)
    
    df_research_pid = df.loc[:,['id','pids', *_EXTRACTED_META_COLS]]
    df_research_pid.dropna(inplace=True)
    
    df_research_pid = df_research_pid.explode('pids').reset_index(drop=True)

    df_pid = pd.json_normalize(df_research_pid['pids'])
    
    df_research_pid = pd.concat(
        [df_research_pid[['id', *_EXTRACTED_META_COLS]].reset_index(drop=True), df_pid.reset_index(drop=True)],
        axis=1,
    )
    df_research_pid = _add_openaire_loaded_metadata(df_research_pid)

    return df_research_pid


In [ ]:
df_research_pid = load_openaire_researchproduct_pids(df_researchproduct_raw)


In [ ]:
pd.DataFrame([{'dataset': 'df_research_pid', 'rows': len(df_research_pid), 'columns': len(df_research_pid.columns)}])


In [ ]:
df_research_pid.head(2)
